In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")


# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature




from myPythonTools.Utils import constants as Co
from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti 
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( puti )
importlib.reload( uti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)
importlib.reload(LL)

grav=Co.grav()


In [ ]:
import pandas as pd

# -------------------------------------------------
# 1. Open dataset
# -------------------------------------------------
Lvn = xr.open_dataset("/glade/work/rneale/data/Livneh/years/prec.2003.nc")

# -------------------------------------------------
# 2. Define landfall date (EDIT if needed)
# -------------------------------------------------
landfall_date = pd.Timestamp("2003-09-19")  # change if needed

start_date = landfall_date - pd.Timedelta(days=2)
end_date   = landfall_date + pd.Timedelta(days=2)

# -------------------------------------------------
# 3. Subset time window
# -------------------------------------------------
Lvn_time = Lvn.sel(time=slice(start_date, end_date))


# -------------------------------------------------
# 5. Accumulate precipitation over 5-day window
# -------------------------------------------------
Acc_precip = Lvn_time["prec"].sum(dim="time")
acc_precip = Acc_precip.values

latP=Lvn.lat.values
lonP=Lvn.lon.values


In [ ]:
#           field2,lat2,lon2 = overplot['field'],overplot['lat'],overplot['lon']

overplot={ 'field':acc_precip , 'lat':latP , 'lon':lonP }

In [ ]:

f1='/glade/work/juliob/HiRes_ana_dev/Data/mpasa3p75_topo_ON_UHR_SEUS.nc'
f2='/glade/work/juliob/HiRes_ana_dev/Data/ne120pg3_topo_ON_UHR_SEUS.nc'
Topo1=xr.open_dataset( f1 )
Topo2=xr.open_dataset( f2 )



h1=Topo1.PHIS.values/grav
h2=Topo2.PHIS.values/grav

lat=Topo1.lat.values
lon=Topo1.lon.values


In [ ]:
cmap='terrain'
levels=[-100,0,1,10,20,30,40,50,70,100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1500]

fields=[h1,h2]
lons=[lon,lon]
lats=[lat,lat]
clevs=[levels,levels]
titles=['3.75 km','14 km']
cmaps=['terrain','terrain']
insets=[]


overplot={ 'field':acc_precip , 'lat':latP , 'lon':lonP }

####

MapProj =ccrs.Orthographic(-75, 35)
                           
map_extent = None #[-140,-0.1,-50,15]
map_extent = [-90,-70,35,45]
map_extent = [-85,-75,35,42]

clevels = [50,55,60,65,70,75,80,85,90,100,120]
clevels = [75,100,125,150,175,200,225,250]
overplot['clevels']=clevels
overplot['alpha']=0.3
overplot['cmap'] ='RdPu'

titles=['3.75 km model topo.','25 km model topo.']


LL.Maps( fields=fields ,lons=lons ,lats=lats ,clevs=clevs,cmaps=cmaps , titles=titles, insets=insets, Projection=MapProj, Landmask=False ,map_extent=map_extent  , coastline_res='high' ,
        overplot=overplot , fig_xsize_per_panel=13.)

In [ ]:
cmap='terrain'
levels=   np.linspace(50,250,num=51) #[75,100,125,150,175,200,225,250]


fields=[acc_precip, acc_precip]
lons=[lonP,lonP]
lats=[latP,latP]
clevs=[levels,levels]
titles=['3.75 km','14 km']
cmaps=['Greys','Greys']
insets=[]
cbarlabels = ["Accum. Precipitation (mm)" , "Accum. Precipitation (mm)" ]

overplot={ 'field':[h2,h1] , 'lat':lat , 'lon':lon }
clevels = [-100,0,1,10,20,30,40,50,70,100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1500]
clevels = [-100,1,50,100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1500]
overplot['clevels']=clevels
overplot['alpha']=0.02
overplot['cmap'] ='terrain'

####

MapProj =ccrs.Orthographic(-75, 35)
                           
map_extent = None #[-140,-0.1,-50,15]
map_extent = [-90,-70,35,45]
map_extent = [-81,-75,35,39]


titles=['25 km model topo.'  ,  '3.75 km model topo.',]

print( cbarlabels)
LL.Maps( fields=fields ,lons=lons ,lats=lats ,clevs=clevs,cmaps=cmaps , titles=titles, insets=insets, Projection=MapProj, Landmask=False ,map_extent=map_extent  , coastline_res='high' ,
        overplot=overplot , cbarlabels=cbarlabels, fig_xsize_per_panel=13.)

In [ ]:
import pandas as pd

# -------------------------------------------------
# 1. Open dataset
# -------------------------------------------------
ds = xr.open_dataset("/glade/work/rneale/data/Livneh/years/prec.2003.nc")

# -------------------------------------------------
# 2. Define landfall date (EDIT if needed)
# -------------------------------------------------
landfall_date = pd.Timestamp("2003-05-19")  # change if needed

start_date = landfall_date - pd.Timedelta(days=2)
end_date   = landfall_date + pd.Timedelta(days=2)

# -------------------------------------------------
# 3. Subset time window
# -------------------------------------------------
ds_time = ds.sel(time=slice(start_date, end_date))

# -------------------------------------------------
# 4. Subset SC to NY region
#    SC ~32N, NY ~41N
#    Livneh longitudes are 0–360
# -------------------------------------------------
lon = ds.lon

if lon.max() > 180:
    lon_min = 360 - 82   # 82W
    lon_max = 360 - 70   # 70W
else:
    lon_min = -82
    lon_max = -70

ds_region = ds_time.sel(
    lat=slice(32, 41),
    lon=slice(lon_min, lon_max)
)

# -------------------------------------------------
# 5. Accumulate precipitation over 5-day window
# -------------------------------------------------
acc_precip = ds_region["prec"].sum(dim="time")

# -------------------------------------------------
# 6. Plot
# -------------------------------------------------
plt.figure(figsize=(8,6))

acc_precip.plot(cmap="viridis")

plt.title(
    "Accumulated Precipitation\n"
    f"{start_date.date()} to {end_date.date()} (±2 days around landfall)"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
ds

In [ ]:
####
if (fld == 'ICEFRAC'):
    MapProj =ccrs.Orthographic(-45, 90)
                               
    map_extent =[10,150,0,50]
    map_extent = None #[-140,-0.1,-50,15]
    
    LL.Maps( fields=fields ,lons=lons ,lats=lats ,clevs=clevs,cmaps=cmaps , titles=titles, insets=insets, Projection=MapProj, Landmask=False ,map_extent=map_extent )
    ####
else:
    MapProj = ccrs.PlateCarree(central_longitude=180.)
    map_extent =[10,150,0,50]
    map_extent = None #[-140,-0.1,-50,15]
    
    LL.Maps( fields=fields ,lons=lons ,lats=lats ,clevs=clevs,cmaps=cmaps , titles=titles, insets=insets, Projection=MapProj, Landmask=False ,map_extent=map_extent , fig_xsize_per_panel=13.   , CoastColors='white' )